# Phase 2 - Step 1: Preprocessing & Data Prep

This notebook handles the preprocessing of raw fundus images and their respective masks.
It produces clean, normalized, augmented image-mask pairs saved to disk and creates train/val/test CSV splits.

**Key Operations:**
1. Circular ROI Masking (Removes black borders)
2. CLAHE (Contrast Enhancement on the Green channel)
3. Resize to $\text{512}\times\text{512}$
4. Save to `data/dataset_stage1_processed/`
5. Generate CSV splits: `train_split.csv`, `val_split.csv`, `test_split.csv`

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

# extract the data from the zip file

In [ ]:
import zipfile


zip_path = "DR dataset stage 1 segmentation.zip"     # Change this to your actual zip file name
extract_dir = "dataset"           # Where to extract your files

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

## 1. Configuration & Paths
Define standard paths based on the project structure.

In [ ]:
# Define base paths
BASE_DATA_DIR = 'dataset'             # Original downloaded data
OUTPUT_DIR = 'dataset_stage1_segmentation_processed'      # Output location

# Input paths
IMG_DIR = os.path.join(BASE_DATA_DIR, 'images')
VESSEL_DIR = os.path.join(BASE_DATA_DIR, 'vessel_masks')
LESION_DIR = os.path.join(BASE_DATA_DIR, 'lesion_masks')

# Output paths
OUT_IMG_DIR = os.path.join(OUTPUT_DIR, 'images')
OUT_VESSEL_DIR = os.path.join(OUTPUT_DIR, 'vessel_masks')
OUT_LESION_DIR = os.path.join(OUTPUT_DIR, 'lesion_masks')

# Target Size
TARGET_SIZE = (512, 512)

# Create output directories if they don't exist
os.makedirs(OUT_IMG_DIR, exist_ok=True)
os.makedirs(OUT_VESSEL_DIR, exist_ok=True)
os.makedirs(OUT_LESION_DIR, exist_ok=True)

# Important lesion subdirectories based on README
LESION_TYPES = ['MA', 'HE', 'EX', 'SE', 'OD', 'merged']
for l_type in LESION_TYPES:
    os.makedirs(os.path.join(OUT_LESION_DIR, l_type), exist_ok=True)

## 2. Core Image Processing Functions
Functions for reading, cropping the circular ROI, and applying CLAHE.

In [ ]:
def crop_roi(img, tol=7):
    """
    Crops the black borders from a fundus image based on a threshold tolerance.
    Returns the cropped image and bounding box coordinates for applying to masks.
    """
    if img.ndim == 2:
        mask = img > tol
    elif img.ndim == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray > tol
    else:
        raise ValueError("Unsupported image dimensions")

    # Find bounding box
    m, n = mask.shape
    mask0, mask1 = mask.any(0), mask.any(1)

    # Check if image is completely black
    if not mask0.any() or not mask1.any():
        return img, (0, m, 0, n)

    col_start, col_end = mask0.argmax(), n - mask0[::-1].argmax()
    row_start, row_end = mask1.argmax(), m - mask1[::-1].argmax()

    return img[row_start:row_end, col_start:col_end], (row_start, row_end, col_start, col_end)

def apply_clahe(img):
    """
    Apply CLAHE to the L channel of LAB color space to enhance contrast
    (Vessels become more distinct).
    """
    # Convert RGB to LAB
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l_channel, a, b = cv2.split(lab)

    # Apply CLAHE to L-channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l_channel)

    # Merge and convert back to RGB
    limg = cv2.merge((cl, a, b))
    final = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
    return final

def process_image(img_path):
    """ Read an image, convert to RGB, crop ROI, apply CLAHE, resize. """
    img = cv2.imread(img_path)
    if img is None:
        return None, None

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 1. Crop ROI
    cropped_img, bbox = crop_roi(img)

    # 2. Apply CLAHE
    enhanced_img = apply_clahe(cropped_img)

    # 3. Resize
    final_img = cv2.resize(enhanced_img, TARGET_SIZE, interpolation=cv2.INTER_LINEAR)

    # Normalize pixel values [0, 1] if needed (done later in PyTorch Dataset, keeping [0, 255] for saving disk space)
    return final_img, bbox

def process_mask(mask_path, bbox):
    """
    Read a mask (binary/grayscale), apply the same crop bounding box, resize.
    Using INTER_NEAREST to preserve binary 0/1 (or 0/255) targets.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None

    # Crop using the same bounding box as the main image
    row_start, row_end, col_start, col_end = bbox
    cropped_mask = mask[row_start:row_end, col_start:col_end]

    # Resize
    final_mask = cv2.resize(cropped_mask, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)

    # Ensure binary format (0 or 255)
    _, final_mask = cv2.threshold(final_mask, 127, 255, cv2.THRESH_BINARY)

    return final_mask


## 3. Main Processing Loop
Iterate through all images, process them, and process corresponding masks.

In [ ]:
# ============================================================
# Fixed Matching Logic
# ============================================================
import os
from glob import glob

def find_vessel_mask(img_name, vessel_dir):
    """
    Find the matching vessel mask for a given image name.
    Handles all naming conventions in our dataset.
    """
    name_no_ext = os.path.splitext(img_name)[0]

    # Strategy 1: Exact stem match (any extension) — works for MAPLES, VESSEL
    for ext in ['.png', '.tif', '.jpg', '.bmp']:
        candidate = os.path.join(vessel_dir, name_no_ext + ext)
        if os.path.exists(candidate):
            return candidate

    # Strategy 2: Replace '_image_' with '_mask_' — works for RETINOMIX
    if '_image_' in name_no_ext:
        mask_stem = name_no_ext.replace('_image_', '_mask_')
        for ext in ['.png', '.tif', '.jpg', '.bmp']:
            candidate = os.path.join(vessel_dir, mask_stem + ext)
            if os.path.exists(candidate):
                return candidate

    return None  # No vessel mask found (e.g., IDRID)


def find_lesion_mask(img_name, lesion_dir, lesion_type):
    """
    Find the matching lesion mask for a given image and lesion type.
    Handles IDRID suffix convention (_EX, _HE, etc.) and MAPLES same-name convention.
    """
    name_no_ext = os.path.splitext(img_name)[0]

    # Strategy 1: Exact stem match (MAPLES style)
    for ext in ['.png', '.tif', '.jpg', '.bmp']:
        candidate = os.path.join(lesion_dir, lesion_type, name_no_ext + ext)
        if os.path.exists(candidate):
            return candidate

    # Strategy 2: Stem + _TYPE suffix (IDRID style: IDRiD_55_EX.tif)
    suffixed = name_no_ext + '_' + lesion_type
    for ext in ['.tif', '.png', '.jpg', '.bmp']:
        candidate = os.path.join(lesion_dir, lesion_type, suffixed + ext)
        if os.path.exists(candidate):
            return candidate

    return None


# ============================================================
# Main Processing Loop (Fixed)
# ============================================================
all_images = glob(os.path.join(IMG_DIR, '*.*'))
print(f"Found {len(all_images)} images in {IMG_DIR}.")

LESION_TYPES = ['MA', 'HE', 'EX', 'SE', 'OD', 'NV', 'CW', 'merged']
for l_type in LESION_TYPES:
    os.makedirs(os.path.join(OUT_LESION_DIR, l_type), exist_ok=True)

valid_records = []
skipped_no_mask = []
vessel_count = 0
lesion_count = 0

for img_path in tqdm(all_images, desc="Processing Dataset"):
    base_name = os.path.basename(img_path)
    name_no_ext = os.path.splitext(base_name)[0]

    # Process the image
    processed_img, bbox = process_image(img_path)
    if processed_img is None:
        continue

    # --- Find vessel mask ---
    vessel_mask_path = find_vessel_mask(base_name, VESSEL_DIR)
    has_vessel = False
    out_vmask_name = name_no_ext + '.png'  # Standardize output to .png

    if vessel_mask_path is not None:
        processed_vmask = process_mask(vessel_mask_path, bbox)
        if processed_vmask is not None:
            out_vmask_path = os.path.join(OUT_VESSEL_DIR, out_vmask_name)
            cv2.imwrite(out_vmask_path, processed_vmask)
            has_vessel = True
            vessel_count += 1

    # --- Find lesion masks ---
    has_any_lesion = False
    for l_type in LESION_TYPES:
        l_mask_path = find_lesion_mask(base_name, LESION_DIR, l_type)
        if l_mask_path is not None:
            processed_lmask = process_mask(l_mask_path, bbox)
            if processed_lmask is not None:
                out_lmask_path = os.path.join(OUT_LESION_DIR, l_type, out_vmask_name)
                cv2.imwrite(out_lmask_path, processed_lmask)
                has_any_lesion = True
                lesion_count += 1

    # Save the processed image if it has ANY mask (vessel or lesion)
    if has_vessel or has_any_lesion:
        out_img_path = os.path.join(OUT_IMG_DIR, out_vmask_name)
        cv2.imwrite(out_img_path, cv2.cvtColor(processed_img, cv2.COLOR_RGB2BGR))

        record = {
            'img_id': out_vmask_name,
            'img_path': os.path.relpath(out_img_path, OUTPUT_DIR),
            'has_vessel': has_vessel,
            'has_lesion': has_any_lesion,
        }
        if has_vessel:
            record['vessel_path'] = os.path.relpath(
                os.path.join(OUT_VESSEL_DIR, out_vmask_name), OUTPUT_DIR
            )
        valid_records.append(record)
    else:
        skipped_no_mask.append(base_name)

print(f"\n✅ Successfully processed: {len(valid_records)} images")
print(f"   - With vessel mask:  {vessel_count}")
print(f"   - With lesion masks: {lesion_count} (across all types)")
print(f"❌ Skipped (no mask at all): {len(skipped_no_mask)}")


## 4. Train / Val / Test Split
We split the processed data into 70% Train, 15% Validation, 15% Test.

In [ ]:
# ============================================================
# Train / Val / Test Split (Fixed)
# ============================================================
if len(valid_records) > 0:
    df = pd.DataFrame(valid_records)

    # Separate: images with vessels vs lesion-only
    vessel_df = df[df['has_vessel'] == True].copy()
    lesion_only_df = df[(df['has_vessel'] == False) & (df['has_lesion'] == True)].copy()

    print(f"Total images:        {len(df)}")
    print(f"  With vessel masks: {len(vessel_df)}  (for vessel segmentation)")
    print(f"  Lesion-only:       {len(lesion_only_df)}  (IDRID, for lesion segmentation)")

    # Split VESSEL images (70/15/15) — used for vessel segmentation training
    train_v, rem_v = train_test_split(vessel_df, test_size=0.30, random_state=42)
    val_v, test_v = train_test_split(rem_v, test_size=0.50, random_state=42)

    print(f"\nVessel split: Train={len(train_v)} | Val={len(val_v)} | Test={len(test_v)}")

    train_v.to_csv(os.path.join(OUTPUT_DIR, 'train_split.csv'), index=False)
    val_v.to_csv(os.path.join(OUTPUT_DIR, 'val_split.csv'), index=False)
    test_v.to_csv(os.path.join(OUTPUT_DIR, 'test_split.csv'), index=False)

    # Save ALL images (vessel + lesion-only) for future lesion segmentation
    df.to_csv(os.path.join(OUTPUT_DIR, 'all_images.csv'), index=False)

    print("✅ CSV files saved!")


## 5. Visualization Sanity Check
Let's visualize a random pair from the output to ensure correctness.

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt

# Number of examples you want to show
num_samples = 5

# Randomly sample rows from your dataframe
samples = df.sample(num_samples)

plt.figure(figsize=(12, num_samples * 4))

for i, (_, row) in enumerate(samples.iterrows()):
    # Read image
    img_path = os.path.join(OUTPUT_DIR, row['img_path'])
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Read mask
    mask_path = os.path.join(OUTPUT_DIR, row['vessel_path'])
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # Show image
    plt.subplot(num_samples, 2, 2*i + 1)
    plt.imshow(img)
    plt.title(f"Image: {row['img_path']}")
    plt.axis("off")

    # Show mask
    plt.subplot(num_samples, 2, 2*i + 2)
    plt.imshow(mask, cmap="gray")
    plt.title(f"Mask: {row['vessel_path']}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt

def show_preprocessing_examples(df, raw_dir, processed_dir, num_samples=5):
    """
    Shows original vs. preprocessed images side by side.
    Assumes raw images use the same filename as processed images.
    """

    samples = df.sample(num_samples)

    plt.figure(figsize=(12, num_samples * 4))

    for i, (_, row) in enumerate(samples.iterrows()):
        # Extract only the filename
        filename = os.path.basename(row["img_path"])

        # Try possible raw extensions
        possible_exts = [".jpg", ".jpeg", ".png", ".tif"]

        raw_img = None
        for ext in possible_exts:
            raw_path = os.path.join(raw_dir, filename.replace(".png", ext))
            if os.path.exists(raw_path):
                raw_img = cv2.imread(raw_path)
                break

        if raw_img is None:
            print(f"Raw file not found for {filename}")
            continue

        raw_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)

        # Load preprocessed image
        proc_path = os.path.join(processed_dir, row["img_path"])
        proc_img = cv2.imread(proc_path)
        proc_img = cv2.cvtColor(proc_img, cv2.COLOR_BGR2RGB)

        # Show original
        plt.subplot(num_samples, 2, 2*i + 1)
        plt.imshow(raw_img)
        plt.title("Original")
        plt.axis("off")

        # Show processed
        plt.subplot(num_samples, 2, 2*i + 2)
        plt.imshow(proc_img)
        plt.title("Preprocessed (512×512)")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

RAW_DIR = os.path.join(BASE_DATA_DIR, "images")

show_preprocessing_examples(
    df=df,
    raw_dir=RAW_DIR,
    processed_dir=OUTPUT_DIR,
    num_samples=5
)

In [ ]:
# ============================================================
# Visualize MAPLES Image + Vessel + Lesion Masks
# ============================================================
import os
import cv2
import random
import matplotlib.pyplot as plt

LESION_TYPES = ['MA', 'HE', 'EX', 'SE', 'OD', 'NV', 'CW', 'merged']
LESION_COLORS = {
    'MA': 'Reds',        # Microaneurysms (red dots)
    'HE': 'Oranges',     # Hemorrhages
    'EX': 'YlOrBr',      # Hard Exudates (yellowish)
    'SE': 'Purples',     # Soft Exudates
    'OD': 'Blues',       # Optic Disc
    'NV': 'Greens',      # Neovascularization
    'CW': 'PuRd',        # Cotton Wool spots
    'merged': 'gray',    # All merged
}

def visualize_maples_pairs(output_dir, num_samples=3):
    """Show random MAPLES images with their vessel and lesion masks."""

    # 1. Filter ONLY for MAPLES images
    all_imgs = [f for f in os.listdir(os.path.join(output_dir, 'images')) if f.startswith('MAPLES')]

    if not all_imgs:
        print("No MAPLES images found in the output directory!")
        return

    samples = random.sample(all_imgs, min(num_samples, len(all_imgs)))

    for img_name in samples:
        img_path = os.path.join(output_dir, 'images', img_name)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 2. Find which lesion masks exist for this specific image
        available_lesions = []
        for lt in LESION_TYPES:
            mask_path = os.path.join(output_dir, 'lesion_masks', lt, img_name)
            if os.path.exists(mask_path):
                available_lesions.append(lt)

        # 3. Check for vessel mask
        vessel_path = os.path.join(output_dir, 'vessel_masks', img_name)
        has_vessel = os.path.exists(vessel_path)

        # 4. Setup Plot Grid (Original + Vessel + each found Lesion)
        ncols = 1 + (1 if has_vessel else 0) + len(available_lesions)
        fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 4))

        # Handle case where ncols=1 (axes isn't an array)
        if ncols == 1:
            axes = [axes]

        fig.suptitle(f"MAPLES Sample: {img_name}", fontsize=14, fontweight='bold')

        col = 0

        # --- Plot Original Image ---
        axes[col].imshow(img)
        axes[col].set_title('Original Image')
        axes[col].axis('off')
        col += 1

        # --- Plot Vessel Mask ---
        if has_vessel:
            vmask = cv2.imread(vessel_path, cv2.IMREAD_GRAYSCALE)
            axes[col].imshow(vmask, cmap='gray')
            axes[col].set_title('Vessels')
            axes[col].axis('off')
            col += 1

        # --- Plot Lesion Masks ---
        for lt in available_lesions:
            mask_path = os.path.join(output_dir, 'lesion_masks', lt, img_name)
            lmask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            # Use specific colormaps to easily distinguish lesions
            cmap = LESION_COLORS.get(lt, 'gray')
            axes[col].imshow(lmask, cmap=cmap)
            axes[col].set_title(f'Lesion: {lt}')
            axes[col].axis('off')
            col += 1

        plt.tight_layout()
        plt.show()

# Run it! (Replace OUTPUT_DIR with your actual processed data folder variable if different)
# e.g., OUTPUT_DIR = '../data/dataset_stage1_processed'
visualize_maples_pairs(OUTPUT_DIR, num_samples=4)


# download the dataset

In [ ]:
import shutil

folder_path = "dataset_stage1_segmentation_processed"   # <-- your folder name
output_zip = "dataset_preprocessed_V2.zip"

shutil.make_archive("dataset_preprocessed_V2", 'zip', folder_path)

print("ZIP created:", output_zip)